# 15 · Custom models and metrics

A model or a metric of your own is a class in a Python file of your own. WarpRec
imports the file, the class registers itself under a name, and from then on the
name works everywhere a built-in's does: in a configuration, in every pipeline,
next to the built-ins in an `Evaluator`. This guide writes three such files in
`user_code/` and covers:

- `general.custom_modules`: how WarpRec imports your files, and the names it
  gives them;
- a closed-form model, fitted in its constructor, whose `predict` serves both
  full and sampled evaluation;
- an iterative model: `get_dataloader`, `forward`, `training_step`, `predict`,
  trained by Lightning;
- a parameter class, so a configuration's hyperparameters are validated, and
  what happens without one;
- a metric, used bare under `metrics` and with a parameter under
  `complex_metrics`;
- all of it run by the design pipeline, then from Python next to the built-ins
  the two models re-implement, which they match exactly.

The [Extending WarpRec](https://warprec.readthedocs.io/en/latest/extending/models/)
pages describe the same interfaces in prose.

**Requirements:** `pip install warprec jupyter`. Runs in one to two minutes on a
laptop CPU. It reads MovieLens-100K through `guide_data.movielens_100k()`, which
downloads and converts it exactly as guide 1 shows.

In [1]:
import inspect
import logging
import re
import sys
import warnings
from pathlib import Path

import lightning as L
import pandas as pd
import torch
import yaml
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

folder = movielens_100k()

# Lightning reports the hardware it found and warns about loader workers; none
# of it is about WarpRec, so it is kept out of the outputs.
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="lightning")
warnings.filterwarnings("ignore", module="torch")
warnings.filterwarnings("ignore", message='install "ipywidgets"')
logger.disable("warprec")


def show(obj):
    """Print the source of a class or function without its docstrings."""
    source = inspect.getsource(obj)
    print(re.sub(r'\n\s*""".*?"""', "", source, flags=re.S))

## Where the code lives

The guide's folder holds a `user_code/` directory with three ordinary Python
files: two models, the parameter class of one of them, and a metric.

In [2]:
sorted(p.name for p in Path("user_code").glob("*.py"))

['my_metrics.py', 'my_models.py', 'my_params.py']

A configuration names them under `general.custom_modules`, as paths relative to
the directory WarpRec is started from (absolute paths work too). Each entry is a
`.py` file or a directory with an `__init__.py`. This is the configuration the
design pipeline runs at the end of the guide:

In [3]:
print(Path("configs/design.yml").read_text())

# The two custom models, scored with the built-in metrics and a custom one.
# Paths are relative to the guide's folder.
general:
  device: cpu
  # Imported before any other section is validated, so the names they
  # register can be used below.
  custom_modules:
    - user_code/my_models.py
    - user_code/my_params.py
    - user_code/my_metrics.py
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.2
models:
  MyEASE:
    l2: 200.0
    optimization:
      num_workers: 0
  MyBPR:
    embedding_size: 64
    batch_size: 1024
    epochs: 10
    learning_rate: 0.005
    optimization:
      num_workers: 0
evaluation:
  top_k: [10]
  metrics: [nDCG, Precision, RBP]
  complex_metrics:
    - name: RBP
      params:
        persistence: 0.5



Loading a configuration imports its custom modules **before** any section is
validated, because the sections check model and metric names against the
registries. A module is imported under its own name, the file name without
`.py` (`my_models`), with its parent directory put on `sys.path`. So after the
configuration is loaded, the module is an ordinary import:

In [4]:
from warprec.utils.config.config import load_design_configuration

config = load_design_configuration("configs/design.yml")

import my_models
from warprec.utils.registry import metric_registry, model_registry, params_registry

print(Path(my_models.__file__).relative_to(Path.cwd()))
print(
    "MyEASE, MyBPR:",
    [n in model_registry.list_registered() for n in ("MYEASE", "MYBPR")],
)
print("MyBPR parameters:", "MYBPR" in params_registry.list_registered())
print("RBP:", "RBP" in metric_registry.list_registered())

user_code/my_models.py
MyEASE, MyBPR: [True, True]
MyBPR parameters: True
RBP: True


The same import, outside a configuration, is
`warprec.utils.helpers.load_custom_modules(["user_code/my_models.py", ...])`.
Two things follow from how it works:

- Registry names are case-insensitive, and registering a name that already
  exists **replaces** the class silently: a custom model registered as `EASE`
  would take the built-in's place in every configuration. Give yours names of
  their own.
- A module is imported once per process. A second file with the same name
  (another project's `my_models.py`) is not imported at all: Python hands back
  the module it already has. Keep the file names distinct.

The train pipeline (guide 10) runs its trials on Ray workers, which do not
share the driver's `sys.path`. It passes every `custom_modules` entry to Ray as
a `py_modules` entry of the runtime environment, which ships the file to every
worker under that same name, and each worker imports it before building a
model. Nothing changes in the configuration.

## A closed-form model

Every model receives the same constructor arguments: `params` (the model's
section of the configuration), `info` (the dataset's dimensions and what it
carries, from `Dataset.info()`), a `seed`, and, as keyword arguments, everything
a pipeline has: the training `interactions`, `sessions`, `transactions`, the
knowledge graph... A model names what it needs and lets `**kwargs` take the
rest. Guide 9 covers the shared constructor in full.

`MyEASE` re-implements EASE. Its hyperparameters are its **class
annotations**: `l2: float` is what makes `Recommender.__init__` copy
`params["l2"]` to `self.l2` (only the class's own annotations are read, so a
subclass re-declares any it inherits). The model is fitted in the constructor;
the result is kept as a buffer, so it moves with the model to a device and is
saved with its state.

In [5]:
show(my_models.MyEASE)

@model_registry.register(name="MyEASE")
class MyEASE(Recommender):

    l2: float

    def __init__(
        self,
        params: dict,
        info: dict,
        interactions: Interactions,
        *args: Any,
        seed: int = 42,
        **kwargs: Any,
    ):
        super().__init__(params, info, *args, seed=seed, **kwargs)
        self.train_matrix = interactions.get_sparse()

        X = self.train_matrix
        gram = (X.T @ X).toarray() + self.l2 * np.identity(self.n_items)
        weights = np.linalg.inv(gram)
        weights /= -np.diag(weights)
        np.fill_diagonal(weights, 0.0)

        # A buffer moves with the model to its device and is saved in its state.
        self.register_buffer("weights", torch.from_numpy(weights))

    def predict(
        self,
        user_indices: Tensor,
        *args: Any,
        item_indices: Optional[Tensor] = None,
        **kwargs: Any,
    ) -> Tensor:
        history = self.train_matrix[user_indices.tolist()].toarray()
       

`predict` is called by the evaluator with a batch of user indices, and with
`item_indices` in one of two shapes:

- `None` under **full** evaluation: return a score for every item,
  `[batch, n_items]`. The evaluator removes the training items afterwards.
- `[batch, candidates]` under **sampled** evaluation: return one score per
  candidate, in the same order. Candidate lists of different lengths are padded
  with the index `n_items`, which is not an item: whatever score the model gives
  it is replaced by −∞, but indexing with it must not fail.

`ItemSimRecommender`, the base class the built-in EASE uses, already provides
this `predict` for any model that fills an item × item matrix; `MyEASE` writes
it out to show what a `predict` has to do.

Built from the dataset the configuration describes, next to the built-in:

In [6]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback

dataset, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
n_users, n_items = dataset.get_dims()

ease = model_registry.get(
    "EASE", params={"l2": 200.0}, info=dataset.info(), interactions=dataset.train_set
)
my_ease = model_registry.get(
    "MyEASE",
    params=config.models["MyEASE"],
    info=dataset.info(),
    interactions=dataset.train_set,
)
print(f"{n_users} users, {n_items} items; MyEASE params: {my_ease.get_params()}")

943 users, 1613 items; MyEASE params: {'l2': 200.0}


Both ways of calling `predict`, for every user. The sampled call takes its
candidates from the first batch of the sampled evaluation loader: each user's
test items and 99 negatives, the test items padded to the longest list.

In [7]:
users = torch.arange(n_users)
sampled_loader = dataset.get_sampled_evaluation_dataloader(num_negatives=99, seed=42)
batch_users, positives, negatives = next(iter(sampled_loader))
candidates = torch.cat([positives, negatives], dim=1)

with torch.inference_mode():
    full = {m.name: m.predict(user_indices=users) for m in (ease, my_ease)}
    sampled = {
        m.name: m.predict(user_indices=batch_users, item_indices=candidates)
        for m in (ease, my_ease)
    }
for kind, scores in [("full", full), ("sampled", sampled)]:
    a, b = scores["EASE"], scores["MyEASE"]
    print(
        f"{kind:8s} shape {tuple(b.shape)}, "
        f"largest difference from EASE {(a - b).abs().max().item():.1e}"
    )
print("padding in the candidates:", int((candidates == n_items).sum()))

full     shape (943, 1613), largest difference from EASE 3.1e-15
sampled  shape (943, 227), largest difference from EASE 7.2e-16
padding in the candidates: 100409


The scores agree to floating-point rounding: the built-in multiplies a sparse
history by the weights, `MyEASE` a dense one. Most of the candidate slots are
padding, because a few users have many more test items than the rest.

## An iterative model

An iterative model is a `LightningModule` as well as a `Recommender`:
`IterativeRecommender` adds the optimiser (built from `learning_rate` and the
`optimization.optimizer` section) and leaves four methods to write. `MyBPR` is
matrix factorisation with the BPR loss:

- `__init__` creates the parameters. The item table has `n_items + 1` rows, the
  last being the padding index, so sampled candidates can be looked up
  directly.
- `get_dataloader` returns the training loader. `Interactions` and `Sessions`
  build the common ones (guide 8); `get_contrastive_dataloader` yields
  `(user, positive, negative)` triples. The pipelines pass loader options
  (workers, pinned memory) as `**kwargs`, so they are passed on.
- `forward` scores pairs; `training_step` returns the batch's loss, and
  Lightning does the backward pass and the optimiser step.
- `predict` serves full and sampled evaluation as above.

`epochs`, `batch_size` and `learning_rate` must be annotated: the pipelines read
`model.epochs` to build the trainer.

In [8]:
show(my_models.MyBPR.__init__)
show(my_models.MyBPR.get_dataloader)
show(my_models.MyBPR.training_step)
show(my_models.MyBPR.predict)

    def __init__(
        self,
        params: dict,
        info: dict,
        *args: Any,
        seed: int = 42,
        **kwargs: Any,
    ):
        super().__init__(params, info, *args, seed=seed, **kwargs)

        # Item n_items is the padding index of candidate lists and sequences.
        self.user_embedding = nn.Embedding(self.n_users, self.embedding_size)
        self.item_embedding = nn.Embedding(
            self.n_items + 1, self.embedding_size, padding_idx=self.n_items
        )
        self.apply(self._init_weights)
        self.bpr_loss = BPRLoss()
        self.epoch_losses: List[float] = []

    def get_dataloader(
        self, interactions: Interactions, sessions: Sessions, **kwargs: Any
    ) -> DataLoader:
        return interactions.get_contrastive_dataloader(
            batch_size=self.batch_size, **kwargs
        )

    def training_step(self, batch: Any, batch_idx: int) -> Tensor:
        user, positive, negative = batch
        return self.bpr_loss(self.f

`training_step` only returns the loss. `IterativeRecommender` logs that loss
as `train_loss`, its mean over the batches of the epoch, for every iterative
model: it is among the metrics of every epoch, and it is what `early_stopping`
with `monitor: loss` and the `ReduceLROnPlateau` scheduler watch (guide 10). `MyBPR`
reads it at the end of every epoch, keeps it, and writes it to the WarpRec log,
through Lightning's `on_train_epoch_end` hook:

In [9]:
show(my_models.MyBPR.on_train_epoch_end)

    def on_train_epoch_end(self):
        self.epoch_losses.append(self.trainer.callback_metrics["train_loss"].item())
        logger.msg(
            f"{self.name} epoch {self.current_epoch + 1}: "
            f"loss {self.epoch_losses[-1]:.4f}"
        )



The pipelines train an iterative model with a `lightning.Trainer` over
`model.get_dataloader(...)` (guide 9 lists what they add). Here a bare trainer
trains `MyBPR` and the built-in `BPR` with `reg_weight: 0`, which is what
`MyBPR` leaves out, with the same seed:

In [10]:
bpr_params = {**config.models["MyBPR"], "reg_weight": 0.0}


def train(model):
    """Train an iterative model for model.epochs epochs, without evaluation."""
    trainer = L.Trainer(
        max_epochs=model.epochs,
        accelerator="cpu",
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    trainer.fit(model, model.get_dataloader(dataset.train_set, dataset.train_session))
    return model


bpr = train(model_registry.get("BPR", params=bpr_params, info=dataset.info(), seed=42))
my_bpr = train(
    model_registry.get(
        "MyBPR", params=config.models["MyBPR"], info=dataset.info(), seed=42
    )
)
print("MyBPR loss per epoch:", [round(x, 3) for x in my_bpr.epoch_losses])

MyBPR loss per epoch: [0.633, 0.317, 0.242, 0.203, 0.179, 0.163, 0.146, 0.135, 0.125, 0.115]


In [11]:
with torch.inference_mode():
    same_full = torch.equal(bpr.predict(users), my_bpr.predict(users))
    same_sampled = torch.equal(
        bpr.predict(batch_users, item_indices=candidates),
        my_bpr.predict(batch_users, item_indices=candidates),
    )
print("identical scores, full:", same_full, "| sampled:", same_sampled)

identical scores, full: True | sampled: True


The loss falls from 0.633 to 0.115 over the ten epochs, and the two models end
with identical scores: same initial weights (same seed, same construction
order), same batches, same loss. This is the check worth making on any model
you write: re-implement something WarpRec has, and compare.

## Validating hyperparameters

A configuration's model section is checked against the model's **parameter
class**, a pydantic model registered in `params_registry`. `MyBPR` has one,
in `user_code/my_params.py`:

In [12]:
print(Path("user_code/my_params.py").read_text())

from pydantic import field_validator

from warprec.utils.config.common import validate_greater_than_zero
from warprec.utils.config.model_configuration import (
    FLOAT_FIELD,
    INT_FIELD,
    RecomModel,
)
from warprec.utils.registry import params_registry


# WarpRec looks a parameter class up by its own class name, so it must be
# named exactly like the model it validates. Hence a module of its own.
@params_registry.register("MyBPR")
class MyBPR(RecomModel):
    """The hyperparameters of the MyBPR model.

    Each field accepts a single value or a search space (guide 10).

    Attributes:
        embedding_size (INT_FIELD): The size of the user and item factors.
        batch_size (INT_FIELD): The number of training triples per batch.
        epochs (INT_FIELD): The number of passes over the training data.
        learning_rate (FLOAT_FIELD): The optimiser's learning rate.
    """

    embedding_size: INT_FIELD
    batch_size: INT_FIELD
    epochs: INT_FIELD
    learning_rate: FL

Each field takes a single value or a search space, which is why the types are
WarpRec's `INT_FIELD` / `FLOAT_FIELD` and the validator checks every value
(`validate_greater_than_zero` and its siblings in
`warprec.utils.config.common` handle both forms). `RecomModel` adds the
`meta`, `optimization` and `early_stopping` blocks every model accepts.

The class must be **named exactly like the model**: WarpRec finds the class
by its registered name, then checks its own class name against the registry
to decide whether to turn the values into search spaces. A class registered as
`MyBPR` but named `MyBPRParams` passes validation, but its values stay plain
lists that the train pipeline cannot read. Two classes with one name cannot
share a module, hence `my_params.py`.

`MyEASE` has no parameter class. The train pipeline's configuration shows the
difference; it is the design configuration plus a validation split, a `writer`
and lists of values to search:

In [13]:
print(Path("configs/train.yml").read_text())

# The same models for the train pipeline (guide 10). Not run in this guide:
# it is loaded to show how the hyperparameters are validated. On Ray, every
# entry of custom_modules is shipped to the workers and imported there under
# the same name.
general:
  device: cpu
  ray_address: local
  ray_verbose: 0
  custom_modules:
    - user_code/my_models.py
    - user_code/my_params.py
    - user_code/my_metrics.py
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.2
  validation_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  MyEASE:              # no parameter class: loaded, not validated
    l2: [100.0, 200.0]
  MyBPR:               # validated by user_code/my_params.py
    embedding_size: [32, 64]
    batch_size: 1024
    epochs: 10
    learning_rate: 0.005
writer:
  dataset_name: ml-100k
  writing_m

Loading it validates every model. For `MyEASE` the log says the
hyperparameters are not validated; for `MyBPR` it says nothing:

In [14]:
from warprec.utils.config.config import TrainConfiguration, load_train_configuration

logger.enable("warprec")
train_config = load_train_configuration("configs/train.yml")
logger.disable("warprec")

08-10-2026 12:44:23 - WarpRec - 📝 Reading train configuration file in: configs/train.yml


08-10-2026 12:44:23 - WarpRec - 📝 Reading process completed correctly.


08-10-2026 12:44:23 - WarpRec - 📢 The model MyEASE is registered in the model registry, but not in parameter registry. The model will be loaded but the hyperparameter will not be validated.


Both sections still come out in the form the train pipeline searches over, a
search space per hyperparameter (`grid` by default):

In [15]:
{name: {k: v for k, v in section.items() if k in ("l2", "embedding_size", "epochs")}
 for name, section in train_config.models.items()}  # fmt: skip

{'MyEASE': {'l2': [<SearchSpace.GRID: 'grid'>, 100.0, 200.0]},
 'MyBPR': {'embedding_size': [<SearchSpace.GRID: 'grid'>, 32, 64],
  'epochs': [<SearchSpace.GRID: 'grid'>, 10]}}

The difference is in what gets through. A negative learning rate for `MyBPR`
stops the configuration from loading; a negative `l2` for `MyEASE` is accepted,
and would only show as a broken model:

In [16]:
raw = yaml.safe_load(Path("configs/train.yml").read_text())
raw["models"]["MyEASE"]["l2"] = -1.0
TrainConfiguration(**raw)
print("MyEASE with l2 = -1: loaded")

raw["models"]["MyBPR"]["learning_rate"] = -0.005
try:
    TrainConfiguration(**raw)
except ValueError as error:
    print([line for line in str(error).splitlines() if "learning_rate" in line][-1])

MyEASE with l2 = -1: loaded
  Value error, Values of learning_rate for MyBPR model must be > 0. Values received as input: [-0.005] [type=value_error, input_value=-0.005, input_type=float]


The design pipeline validates each model through the same class just before
building it, so the bad learning rate fails there too, after the data has been
read. It does not log the warning for a model without a class.

## A custom metric

A metric is a [torchmetrics](https://lightning.ai/docs/torchmetrics/) `Metric`
registered in `metric_registry`. WarpRec's base classes do the accumulation:
a metric averaged over users subclasses `UserAverageTopKMetric` and writes one
method, `compute_scores`, which returns a value per user of the batch. `RBP` is
rank-biased precision, a hit at rank *i* weighted by (1 − p)·p<sup>i−1</sup>:

In [17]:
import my_metrics

show(my_metrics.RBP)

@metric_registry.register("RBP")
class RBP(UserAverageTopKMetric):

    # The blocks compute_scores reads; the evaluator computes each once per batch.
    _REQUIRED_COMPONENTS: Set[MetricBlock] = {
        MetricBlock.BINARY_RELEVANCE,
        MetricBlock.VALID_USERS,
        MetricBlock.TOP_K_BINARY_RELEVANCE,
    }

    def __init__(
        self,
        k: int,
        num_users: int,
        *args: Any,
        persistence: float = 0.8,
        dist_sync_on_step: bool = False,
        **kwargs: Any,
    ):
        super().__init__(k, num_users, *args, dist_sync_on_step=dist_sync_on_step)
        if not 0 < persistence < 1:
            raise ValueError(f"persistence must be in (0, 1), got {persistence}.")
        self.persistence = persistence

    def compute_scores(
        self, preds: Tensor, target: Tensor, top_k_rel: Tensor, **kwargs: Any
    ) -> Tensor:
        ranks = torch.arange(top_k_rel.shape[1], device=top_k_rel.device)
        weights = (1 - self.persistence) * self.

Three parts carry the WarpRec contract:

- `_REQUIRED_COMPONENTS` lists the **metric blocks** the metric reads. The
  evaluator computes the union of all metrics' blocks once per batch and passes
  them to every metric; a block no metric asked for arrives as `None`, so a
  metric must declare every block it reads, even one that another metric in
  the same run already asks for. Guide 12 lists which built-in metric reads
  which block.
- The constructor receives the cutoff `k`, the evaluator's common arguments
  (`num_users`, `num_items`, the training matrix, item counts, clusters, the
  stash...) and the metric's own parameters, all as keywords: name the ones you
  use and let `**kwargs` take the rest.
- `name` is the key the results are reported under. A metric with parameters
  names its non-default variants after them, so `RBP` and `RBP[p=0.5]` can be
  computed side by side.

In a configuration, a metric without parameters goes under `metrics`, and one
with parameters under `complex_metrics` as a `name` and its `params`, exactly
as `configs/design.yml` above does. The `Evaluator` takes the same two lists:

In [18]:
from warprec.evaluation import Evaluator

evaluator = Evaluator(
    list(config.evaluation.metrics),
    list(config.evaluation.top_k),
    train_set=dataset.train_set.get_sparse(),
    complex_metrics=config.evaluation.complex_metrics,
)
evaluator.evaluate(my_ease, dataset.get_evaluation_dataloader(), "full", dataset)
results = {
    name: value.nanmean().item()
    for name, value in evaluator.compute_results()[10].items()
}
results

{'nDCG': 0.20893052220344543,
 'Precision': 0.18154826760292053,
 'RBP': 0.1770738959312439,
 'RBP[p=0.5]': 0.22065341472625732}

A metric deserves the same check as a model. RBP@10 by hand, from `MyEASE`'s
top 10 with the training items removed, agrees with what the evaluator
accumulated:

In [19]:
train_matrix = torch.from_numpy(dataset.train_set.get_sparse().toarray()).bool()
test_matrix = torch.from_numpy(dataset.eval_set.get_sparse().toarray()) > 0
with torch.inference_mode():
    scores = my_ease.predict(torch.arange(n_users)).masked_fill(
        train_matrix, -torch.inf
    )
hits = test_matrix.gather(1, scores.topk(10, dim=1).indices).float()
by_hand = {
    p: ((1 - p) * p ** torch.arange(10.0) * hits).sum(dim=1).mean().item()
    for p in (0.8, 0.5)
}
print(f"RBP@10 by hand {by_hand[0.8]:.6f}, evaluator {results['RBP']:.6f}")
print(
    f"RBP[p=0.5]@10 by hand {by_hand[0.5]:.6f}, evaluator {results['RBP[p=0.5]']:.6f}"
)

RBP@10 by hand 0.177074, evaluator 0.177074
RBP[p=0.5]@10 by hand 0.220653, evaluator 0.220653


(Every user has test items here, so the plain mean and the evaluator's
average over evaluated users coincide.)

## The design pipeline

`design_pipeline` runs `configs/design.yml` end to end: it imports the custom
modules, reads and splits the data, builds each model, trains `MyBPR` with
an evaluation on the test split after every epoch, and logs the test results
with the custom metric in both variants, all in this process. The log is on
for this run; Lightning adds a summary of `MyBPR`'s parameters.

In [20]:
from warprec.pipelines import design_pipeline

logger.enable("warprec")
design_pipeline("configs/design.yml")
logger.disable("warprec")

08-10-2026 12:44:23 - WarpRec - 📝 Starting the Design Pipeline.


08-10-2026 12:44:23 - WarpRec - 📝 Reading design configuration file in: configs/design.yml


08-10-2026 12:44:23 - WarpRec - 📝 Reading process completed correctly.


08-10-2026 12:44:23 - WarpRec - 📝 Starting test splitting process with temporal_holdout splitting strategy.


08-10-2026 12:44:23 - WarpRec - 📝 Test splitting completed in : 0.01s


...

08-10-2026 12:44:31 - WarpRec - 📝 | Cutoff   |     nDCG |   Precision |      RBP |   RBP[p=0.5] |


08-10-2026 12:44:31 - WarpRec - 📝 +==========+==========+=============+==========+==============+


08-10-2026 12:44:31 - WarpRec - 📝 | Top@10   | 0.179576 |    0.157476 | 0.152789 |     0.190807 |


08-10-2026 12:44:31 - WarpRec - 📝 +----------+----------+-------------+----------+--------------+


08-10-2026 12:44:31 - WarpRec - ✅ Design pipeline executed successfully. WarpRec is shutting down.


The `MyBPR epoch` lines are the model's own `on_train_epoch_end`. `MyEASE`'s
results are those of the evaluator above, and `MyBPR`'s are those of the
built-in BPR without regularisation, as the next section shows.

## Next to the built-ins

Custom and built-in models go through the same evaluator. With the models
trained above, under full evaluation and against 99 sampled negatives:

In [21]:
models = {"EASE": ease, "MyEASE": my_ease, "BPR": bpr, "MyBPR": my_bpr}
loaders = {
    "full": ("full", dataset.get_evaluation_dataloader()),
    "sampled": ("sampled", sampled_loader),
}
table = {}
for protocol, (strategy, loader) in loaders.items():
    for name, model in models.items():
        evaluator.evaluate(model, loader, strategy, dataset)
        table[(protocol, name)] = {
            f"{metric}@{k}": value.nanmean().item()
            for k, metrics in evaluator.compute_results().items()
            for metric, value in metrics.items()
        }
pd.DataFrame(table).T.round(4)

nDCG@10  Precision@10  RBP@10  RBP[p=0.5]@10
full    EASE     0.2089        0.1815  0.1771         0.2207
        MyEASE   0.2089        0.1815  0.1771         0.2207
        BPR      0.1796        0.1575  0.1528         0.1908
        MyBPR    0.1796        0.1575  0.1528         0.1908
sampled EASE     0.6660        0.5789  0.5555         0.6655
        MyEASE   0.6660        0.5789  0.5555         0.6655
        BPR      0.6417        0.5596  0.5322         0.6358
        MyBPR    0.6417        0.5596  0.5322         0.6358

Each custom model matches its built-in on every metric under both protocols:
`MyEASE` 0.2089 nDCG@10 like EASE, `MyBPR` 0.1796 like BPR in full evaluation,
the same numbers the design pipeline logged.

## Summary

| what | where |
|---|---|
| `general.custom_modules` | list of `.py` files or package directories, imported by file name before the configuration is validated; shipped to Ray workers by the train pipeline |
| model class | subclass `Recommender` (closed form), `ItemSimRecommender` (item × item matrix) or `IterativeRecommender`; `@model_registry.register(name=...)`; hyperparameters as class annotations |
| `predict` | `item_indices=None` → `[batch, n_items]`; candidates → `[batch, candidates]`, padding index `n_items` |
| iterative models | `get_dataloader`, `forward`, `training_step` (returns the loss), `predict`; annotate `epochs`, `batch_size`, `learning_rate` |
| parameter class | `RecomModel` subclass named like the model, `@params_registry.register(...)`, field validators; without one the train pipeline warns and validates nothing |
| metric class | `UserAverageTopKMetric` + `compute_scores` + `_REQUIRED_COMPONENTS`; `@metric_registry.register(...)`; a `name` for non-default parameters |
| `evaluation.metrics` / `complex_metrics` | a custom metric bare, or with `name` + `params` |

The [Custom models](https://warprec.readthedocs.io/en/latest/extending/models/)
and [Custom metrics](https://warprec.readthedocs.io/en/latest/extending/metrics/)
pages describe the interfaces;
[General configuration](https://warprec.readthedocs.io/en/latest/configuration/general/)
lists `custom_modules`. Guide 16 covers callbacks and pipelines of your own.